# Analysis of ZeroSumNormal Constraint on Fourier Seasonality

This notebook demonstrates a problem with using `ZeroSumNormal` to constrain Fourier coefficients in the `StateSpaceTimeSeries` model. We show that certain valid seasonal patterns **cannot be fitted** due to this constraint.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## Background

The `StateSpaceTimeSeries` model in CausalPy uses `pymc-extras.FrequencySeasonality` for seasonal components. The current implementation constrains the Fourier coefficients with `ZeroSumNormal`:

```python
_annual_seasonal = pm.ZeroSumNormal("params_freq", sigma=80, dims=annual_dims)
```

This enforces $\sum_j (a_j + b_j) = 0$ where $a_j, b_j$ are the cosine and sine coefficients.

## The Fourier Basis

For period $S=12$ (monthly data with annual seasonality), `FrequencySeasonality` uses harmonics $j=1,2,\ldots,6$ with:
- $j=1$ to $5$: both $\cos$ and $\sin$ terms (10 parameters)
- $j=6$ (Nyquist): only $\cos$ term (1 parameter, since $\sin(\pi t) = 0$ for integer $t$)

Total: **11 parameters**

In [ ]:
S = 12  # Period (monthly)
n = S // 2  # Number of harmonics = 6
n_params = 2 * (n - 1) + 1  # 11 parameters

print(f"Period S = {S}")
print(f"Number of harmonics n = {n}")
print(f"Number of parameters = {n_params}")
print(f"\nParameter structure:")
for j in range(1, n):
    print(f"  j={j}: cos(2π·{j}·t/{S}), sin(2π·{j}·t/{S})")
print(f"  j={n}: cos(2π·{n}·t/{S}) [Nyquist, sin excluded]")

In [ ]:
def build_fourier_basis(t, S=12):
    """Build Fourier basis matching pymc-extras FrequencySeasonality."""
    n = S // 2
    basis = []
    for j in range(1, n):  # j = 1 to 5
        basis.append(np.cos(2 * np.pi * j * t / S))
        basis.append(np.sin(2 * np.pi * j * t / S))
    basis.append(np.cos(2 * np.pi * n * t / S))  # Nyquist cos only
    return np.column_stack(basis)

t = np.arange(S)
X = build_fourier_basis(t, S)
print(f"Basis matrix shape: {X.shape}")

## Normal Seasonal Patterns: Successfully Fitted

Let's first show that typical seasonal patterns **can** be fitted, even with the zero-sum constraint.

In [ ]:
def fit_with_constraint(y, X):
    """Fit y = X @ theta with zero-sum constraint on theta."""
    # Unconstrained OLS
    theta_ols, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    # Project onto zero-sum subspace
    theta_zs = theta_ols - np.mean(theta_ols)
    y_fit = X @ theta_zs
    return theta_zs, y_fit

def fit_unconstrained(y, X):
    """Fit y = X @ theta without constraint."""
    theta_ols, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    y_fit = X @ theta_ols
    return theta_ols, y_fit

In [ ]:
# Example 1: Simple sinusoidal pattern
y_simple = 2 * np.cos(2 * np.pi * t / S) + 0.5 * np.sin(2 * np.pi * 2 * t / S)

theta_zs, y_fit_zs = fit_with_constraint(y_simple, X)
theta_ols, y_fit_ols = fit_unconstrained(y_simple, X)

fig, ax = plt.subplots()
ax.plot(t, y_simple, 'ko-', label='Target', markersize=8)
ax.plot(t, y_fit_zs, 'b--', label=f'Zero-sum fit (RMSE={np.sqrt(np.mean((y_simple-y_fit_zs)**2)):.4f})', linewidth=2)
ax.set_xlabel('Month (t)')
ax.set_ylabel('Seasonal effect')
ax.set_title('Example 1: Simple Sinusoidal Pattern')
ax.legend()
ax.set_xticks(t)
plt.tight_layout()
plt.show()

print(f"Sum of coefficients: {np.sum(theta_zs):.6f}")
print(f"Fit RMSE: {np.sqrt(np.mean((y_simple - y_fit_zs)**2)):.6f}")

In [ ]:
# Example 2: Complex multi-harmonic pattern
np.random.seed(42)
theta_random = np.random.randn(n_params)
y_complex = X @ theta_random

theta_zs, y_fit_zs = fit_with_constraint(y_complex, X)

fig, ax = plt.subplots()
ax.plot(t, y_complex, 'ko-', label='Target', markersize=8)
ax.plot(t, y_fit_zs, 'b--', label=f'Zero-sum fit (RMSE={np.sqrt(np.mean((y_complex-y_fit_zs)**2)):.4f})', linewidth=2)
ax.set_xlabel('Month (t)')
ax.set_ylabel('Seasonal effect')
ax.set_title('Example 2: Complex Multi-Harmonic Pattern')
ax.legend()
ax.set_xticks(t)
plt.tight_layout()
plt.show()

print(f"Original coefficient sum: {np.sum(theta_random):.4f}")
print(f"Zero-sum coefficient sum: {np.sum(theta_zs):.6f}")
print(f"Fit RMSE: {np.sqrt(np.mean((y_complex - y_fit_zs)**2)):.6f}")

## The Unrepresentable Signal

Now we construct a signal that **cannot** be fitted by the zero-sum constrained model.

### Closed-Form Definition

The unrepresentable signal $g(t)$ has the closed form:

$$g(t) = \begin{cases} 
n = S/2 & \text{if } t = 0 \\
0 & \text{if } t \text{ even}, t \neq 0 \\
\cot\left(\frac{\pi t}{S}\right) - 1 & \text{if } t \text{ odd}
\end{cases}$$

For $S=12$, this gives $g(0) = 6$.

In [ ]:
def g_unrepresentable(t, S=12):
    """
    Closed-form for the unrepresentable signal.
    
    g(t) = ⎧ n = S/2        if t = 0
           ⎨ 0              if t even, t ≠ 0  
           ⎩ cot(πt/S) - 1  if t odd
    """
    t = np.asarray(t) % S
    result = np.zeros_like(t, dtype=float)
    
    # t = 0
    result[t == 0] = S // 2
    
    # t odd
    odd_mask = (t % 2 == 1)
    result[odd_mask] = 1.0 / np.tan(np.pi * t[odd_mask] / S) - 1.0
    
    # t even, t ≠ 0: already 0
    return result

g = g_unrepresentable(t, S)

print("The unrepresentable signal g(t):")
print("=" * 40)
for ti, gi in zip(t, g):
    if ti == 0:
        formula = f"n = {S//2}"
    elif ti % 2 == 0:
        formula = "0"
    else:
        formula = f"cot(π·{ti}/{S}) - 1"
    print(f"  g({ti:2d}) = {gi:8.4f}  [{formula}]")
print("=" * 40)
print(f"  Mean: {np.mean(g):.6f}")
print(f"  Std:  {np.std(g):.4f}")

In [ ]:
# Visualize the unrepresentable signal
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(t, g, color='crimson', alpha=0.7, edgecolor='black')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Month (t)', fontsize=12)
ax.set_ylabel('g(t)', fontsize=12)
ax.set_title('The Unrepresentable Signal g(t)', fontsize=14)
ax.set_xticks(t)
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])

# Annotate key values
ax.annotate(f'g(0) = {g[0]:.0f}', xy=(0, g[0]), xytext=(1, g[0]+0.5),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=10)
ax.annotate(f'g(11) = {g[11]:.2f}', xy=(11, g[11]), xytext=(9.5, g[11]-1),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=10)

plt.tight_layout()
plt.show()

### Why This Signal Corresponds to θ = (1, 1, ..., 1)

The signal $g(t)$ is constructed so that its Fourier coefficients are all equal to 1. Let's verify:

In [ ]:
# Verify that g corresponds to θ = (1, 1, ..., 1)
theta_ols, _ = fit_unconstrained(g, X)

print("Fourier coefficients recovered by unconstrained OLS:")
print(f"  θ = {np.round(theta_ols, 6)}")
print(f"  All equal to 1? {np.allclose(theta_ols, 1.0)}")
print(f"  Sum of coefficients: {np.sum(theta_ols):.4f}")

## The Fitting Failure

Now let's see what happens when we try to fit $g(t)$ with the zero-sum constraint:

In [ ]:
# Attempt to fit with zero-sum constraint
theta_zs, y_fit_zs = fit_with_constraint(g, X)

print("Zero-sum constrained fit:")
print(f"  θ_zs = {np.round(theta_zs, 6)}")
print(f"  Sum of θ_zs: {np.sum(theta_zs):.10f}")
print(f"  All zeros? {np.allclose(theta_zs, 0)}")
print()
print("Fitted signal:")
print(f"  y_fit = {np.round(y_fit_zs, 6)}")
print(f"  All zeros? {np.allclose(y_fit_zs, 0)}")
print()
print("Residual statistics:")
residual = g - y_fit_zs
print(f"  RMSE: {np.sqrt(np.mean(residual**2)):.4f}")
print(f"  Variance explained: 0%")
print(f"  Variance unexplained: 100%")

In [ ]:
# Visualize the failure
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Target vs Fit
ax = axes[0]
ax.plot(t, g, 'ro-', label='Target g(t)', markersize=10, linewidth=2)
ax.plot(t, y_fit_zs, 'b^--', label='Zero-sum fit (≡ 0)', markersize=8, linewidth=2)
ax.axhline(0, color='gray', linewidth=0.5, linestyle=':')
ax.set_xlabel('Month (t)', fontsize=12)
ax.set_ylabel('Value', fontsize=12)
ax.set_title('Zero-Sum Constrained Model CANNOT Fit g(t)', fontsize=14)
ax.legend(fontsize=11)
ax.set_xticks(t)

# Right: Residual
ax = axes[1]
ax.bar(t, residual, color='crimson', alpha=0.7, edgecolor='black')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Month (t)', fontsize=12)
ax.set_ylabel('Residual', fontsize=12)
ax.set_title(f'Residual = g(t) (100% unexplained, RMSE={np.sqrt(np.mean(residual**2)):.2f})', fontsize=14)
ax.set_xticks(t)

plt.tight_layout()
plt.show()

## Comparison: Constrained vs Unconstrained

The following table compares the fitting results:

In [ ]:
# Create comparison table
theta_ols, y_fit_ols = fit_unconstrained(g, X)
theta_zs, y_fit_zs = fit_with_constraint(g, X)

rmse_ols = np.sqrt(np.mean((g - y_fit_ols)**2))
rmse_zs = np.sqrt(np.mean((g - y_fit_zs)**2))

df_comparison = pd.DataFrame({
    'Metric': ['Sum of θ', 'RMSE', 'Variance Explained', 'Can fit g(t)?'],
    'Unconstrained': [f'{np.sum(theta_ols):.1f}', f'{rmse_ols:.2e}', '100%', 'YES'],
    'Zero-Sum Constrained': [f'{np.sum(theta_zs):.1f}', f'{rmse_zs:.2f}', '0%', 'NO']
})

print(df_comparison.to_string(index=False))

## Why This Matters

The signal $g(t)$ is a **valid zero-mean seasonal pattern**. It represents a seasonal effect with:
- Strong January peak ($g(0) = 6$)
- Strong December trough ($g(11) \approx -4.73$)
- Values at odd months following $\cot(\pi t/12) - 1$
- Zero values at even months (except January)

There is no physical reason why this pattern should be unfittable. The `ZeroSumNormal` constraint arbitrarily excludes it.

## The Constraint Equation

The `ZeroSumNormal` constraint enforces orthogonality to $g(t)$:

$$\boxed{\frac{S}{2}\gamma(0) + \sum_{t \text{ odd}} \gamma(t)\left[\cot\left(\frac{\pi t}{S}\right) - 1\right] = 0}$$

where $\gamma(t)$ is the seasonal effect. This constraint has **no natural physical interpretation**.

## Recommendation

Replace `ZeroSumNormal` with `Normal` in `StateSpaceTimeSeries`:

```python
# Current (problematic)
_annual_seasonal = pm.ZeroSumNormal("params_freq", sigma=80, dims=annual_dims)

# Recommended
_annual_seasonal = pm.Normal("params_freq", mu=0, sigma=80, dims=annual_dims)
```

The zero-mean property of the seasonal effect is **already guaranteed** by the Fourier basis with $j \geq 1$ (no DC component). The `ZeroSumNormal` constraint adds an unnecessary restriction.

In [ ]:
# Verify that Fourier basis with j≥1 already has zero mean
print("Mean of each basis function over one period:")
for i in range(n_params):
    mean_i = np.mean(X[:, i])
    print(f"  Basis {i}: mean = {mean_i:.10f}")
print("\nAll basis functions have zero mean → seasonal effect automatically has zero mean.")